# Лабораторная работа № 5 — учебные примеры

Примеры восстановлены из существующего отчёта ЛР5 и адаптированы в нём к parking. Сначала выполните ЛР3. Дополнительные клиенты создаются только внутри транзакции; последняя ячейка отменяет все временные изменения. При прерывании выполните `conn.rollback()` перед повтором. Если идентификаторы клиентов 9 и 10 заняты, вставка остановится без перезаписи данных.

Выполняйте ячейки сверху вниз (**Run All**). Запросы находятся прямо в ноутбуке; SQL-файлы не читаются. Параметры подключения — в следующей ячейке. Зависимости: `pip install "psycopg[binary]" pandas ipykernel` (также поддерживается psycopg2).

[Все лабораторные](../../README.md) · [Отчёты ЛР5](../../labs/lab05/)

## Содержание

- [1. Отбор строк, сортировка, вычисляемые столбцы, DISTINCT, LIMIT, OFFSET, CASE](#part-1)
- [2. Соединения](#part-2)
- [3. Внешние соединения](#part-3)
- [4. Рефлексивные соединения](#part-4)
- [5. Вложенные запросы](#part-5)
- [6. Пример выполнения задания (стр. 117)](#part-6)

In [1]:
import os
import getpass
import pandas as pd
from IPython.display import display
try:
    import psycopg
except ImportError:
    import psycopg2 as psycopg

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", None)
# При необходимости задайте PGHOST, PGPORT, PGUSER, PGPASSWORD перед запуском.
# Пароль в ноутбуке не сохраняйте: драйвер также поддерживает ~/.pgpass.
DB_NAME = os.getenv("PGDATABASE", "parking")
conn = psycopg.connect(
    host=os.getenv("PGHOST", "127.0.0.1"),
    port=int(os.getenv("PGPORT", "5432")),
    user=os.getenv("PGUSER", getpass.getuser()),
    password=os.getenv("PGPASSWORD") or None,
    dbname=DB_NAME,
)
conn.autocommit = False

def q(sql):
    """Выполнить инструкцию; выборку показать таблицей."""
    try:
        with conn.cursor() as cur:
            cur.execute(sql)
            if cur.description:
                frame = pd.DataFrame(cur.fetchall(), columns=[c[0] for c in cur.description])
                frame.index = range(1, len(frame) + 1)
                display(frame)
                print(f"Строк: {len(frame)}")
            else:
                print(cur.statusmessage)
    except Exception:
        conn.rollback()
        raise

run = q
print("Подключение:", DB_NAME)


Подключение: parking


<a id="part-1"></a>

## 1. Отбор строк, сортировка, вычисляемые столбцы, DISTINCT, LIMIT, OFFSET, CASE

**Запрос 1.1.** Шаблон LIKE: выбрать все тарифы для легковых автомобилей

In [2]:
q("""
SELECT * FROM tariffs WHERE tar_name LIKE '%легковой%';
""")

,tar_id,tar_name,tar_period,tar_price,tar_cat_id,tar_date_from,tar_date_to
1,1,"Часовой, легковой",час,60.00,1,2024-01-01,None
2,2,"Суточный, легковой",сутки,350.00,1,2024-01-01,None
3,3,"Месячный, легковой",месяц,4500.00,1,2024-01-01,2024-12-31
4,7,"Месячный, легковой",месяц,4950.00,1,2025-01-01,None


Строк: 4


**Запрос 1.2.** Шаблон с символом «_»: найти марки транспортных средств, названия которых состоят из четырёх символов

In [3]:
q("""
SELECT * FROM vehicles WHERE veh_brand LIKE '____';
""")

,veh_id,veh_plate,veh_brand,veh_model,veh_color,veh_cat_id,veh_client_id
1,1,А123ВС55,Lada,Vesta,белый,1,1


Строк: 1


**Запрос 1.3.** NOT LIKE: клиенты, адрес электронной почты которых не относится к доменам mail.ru и yandex.ru

In [4]:
q("""
SELECT * FROM clients
WHERE cl_email NOT LIKE '%@mail.ru' AND
      cl_email NOT LIKE '%@yandex.ru';
""")

,cl_id,cl_type,cl_name,cl_doc,cl_phone,cl_email,cl_discount,cl_reg_date
1,3,Ю,ООО «Сибтранс»,5503012345,+73812551020,office@sibtrans.ru,10,2024-02-01
2,5,Ф,Нечаева Елена Борисовна,5202 443311,+79139990015,necheva@gmail.com,5,2024-03-05
3,6,Ю,АО «Омскстрой»,5506078901,+73812445566,info@omskstroy.ru,15,2024-03-12


Строк: 3


**Запрос 1.4.** Регулярное выражение POSIX (~): выбрать транспортные средства марок Lada и Kia

In [5]:
q("""
SELECT * FROM vehicles WHERE veh_brand ~ '^(Lada|Kia)';
""")

,veh_id,veh_plate,veh_brand,veh_model,veh_color,veh_cat_id,veh_client_id
1,1,А123ВС55,Lada,Vesta,белый,1,1
2,2,В456ЕК55,Kia,Rio,серый,1,2


Строк: 2


**Запрос 1.5.** Оператор !~: найти модели, названия которых не оканчиваются цифрой

In [6]:
q("""
SELECT * FROM vehicles WHERE veh_model !~ '[0-9]$';
""")

,veh_id,veh_plate,veh_brand,veh_model,veh_color,veh_cat_id,veh_client_id
1,1,А123ВС55,Lada,Vesta,белый,1,1
2,2,В456ЕК55,Kia,Rio,серый,1,2
3,3,Е789МН55,ГАЗ,Газель,белый,3,3
4,8,Р258ЕК55,Renault,Duster,зелёный,2,8
5,10,Т741ОР55,Volkswagen,Polo,белый,1,2
6,5,М654СТ55,Hyundai,Solaris,серебристый,1,5


Строк: 6


**Запрос 1.6.** Предикат BETWEEN: найти тарифы в диапазоне цен от 4000 до 6000 руб.

In [7]:
q("""
SELECT * FROM tariffs WHERE tar_price BETWEEN 4000 AND 6000;
""")

,tar_id,tar_name,tar_period,tar_price,tar_cat_id,tar_date_from,tar_date_to
1,4,"Месячный, внедорожник",месяц,5500.00,2,2024-01-01,None
2,3,"Месячный, легковой",месяц,4500.00,1,2024-01-01,2024-12-31
3,7,"Месячный, легковой",месяц,4950.00,1,2025-01-01,None


Строк: 3


**Запрос 1.7.** Вычисляемый столбец: показать цену тарифа не только в рублях, но и в долларах (условный курс 90 руб.)

In [8]:
q("""
SELECT tar_name, tar_price, round(tar_price/90, 2) AS USD
FROM tariffs;
""")

,tar_name,tar_price,usd
1,"Часовой, легковой",60.00,0.67
2,"Суточный, легковой",350.00,3.89
3,"Месячный, внедорожник",5500.00,61.11
4,"Месячный, грузовой",9000.00,100.00
5,"Месячный, мотоцикл",2500.00,27.78
6,"Месячный, легковой",4500.00,50.00
7,"Месячный, легковой",4950.00,55.00


Строк: 7


**Запрос 1.8.** ORDER BY: список сотрудников в алфавитном порядке

In [9]:
q("""
SELECT * FROM employees ORDER BY emp_name ASC;
""")

,emp_id,emp_name,emp_position,emp_phone,emp_hire_date,emp_chief_id
1,4,Ветров Илья Романович,охранник,+79131110004,2022-09-12,1.0
2,1,Гордеев Пётр Ильич,администратор,+79131110001,2020-03-01,NaN
3,3,Ким Дмитрий Сергеевич,старший кассир,+79131110003,2022-02-01,1.0
4,5,Пащенко Ольга Ивановна,охранник,+79131110005,2023-04-03,1.0
5,2,Сомова Анна Львовна,кассир,+79131110002,2021-06-15,1.0


Строк: 5


**Запрос 1.9.** DISTINCT: какого цвета транспортные средства стоят на автостоянке

In [10]:
q("""
SELECT DISTINCT veh_color FROM vehicles ORDER BY 1;
""")

,veh_color
1,белый
2,зелёный
3,красный
4,оранжевый
5,серебристый
6,серый
7,чёрный


Строк: 7


**Запрос 1.10.** LIMIT: найти три самых дорогих тарифа

In [11]:
q("""
SELECT tar_ID, tar_name, tar_price
FROM tariffs
ORDER BY tar_price DESC
LIMIT 3;
""")

,tar_id,tar_name,tar_price
1,5,"Месячный, грузовой",9000.00
2,4,"Месячный, внедорожник",5500.00
3,7,"Месячный, легковой",4950.00


Строк: 3


**Запрос 1.11.** OFFSET: найти следующие три тарифа, занимающие места с четвёртого по шестое

In [12]:
q("""
SELECT tar_ID, tar_name, tar_price
FROM tariffs
ORDER BY tar_price DESC
LIMIT 3 OFFSET 3;
""")

,tar_id,tar_name,tar_price
1,3,"Месячный, легковой",4500.00
2,6,"Месячный, мотоцикл",2500.00
3,2,"Суточный, легковой",350.00


Строк: 3


**Запрос 1.12.** Условное выражение CASE: разделить месячные тарифы на категории «Дешёвый», «Средний», «Дорогой»

In [13]:
q("""
SELECT tar_name, tar_price,
       CASE WHEN tar_price <= 3000 THEN 'Дешёвый'
            WHEN tar_price <= 5000 THEN 'Средний'
            ELSE 'Дорогой'
       END AS Категория
FROM tariffs
WHERE tar_period = 'месяц'
ORDER BY tar_ID;
""")

,tar_name,tar_price,Категория
1,"Месячный, легковой",4500.00,Средний
2,"Месячный, внедорожник",5500.00,Дорогой
3,"Месячный, грузовой",9000.00,Дорогой
4,"Месячный, мотоцикл",2500.00,Дешёвый
5,"Месячный, легковой",4950.00,Средний


Строк: 5


<a id="part-2"></a>

## 2. Соединения

**Запрос 1.2.1.** Соединение через WHERE: номера договоров с именами клиентов (первые четыре строки)

In [14]:
q("""
SELECT contracts.contr_number, clients.cl_name
FROM contracts, clients
WHERE contracts.contr_client_ID = clients.cl_ID
ORDER BY contracts.contr_ID
LIMIT 4;
""")

,contr_number,cl_name
1,Д-2024/001,Лебедев Игорь Олегович
2,Д-2024/002,Сафина Марина Рустамовна
3,Д-2024/003,ООО «Сибтранс»
4,Д-2024/004,Круглов Антон Павлович


Строк: 4


**Запрос 1.2.2.** Соединение нескольких таблиц: к каким категориям относится транспортное средство по договору с номером 3

In [15]:
q("""
SELECT vehicle_categories.cat_name
FROM vehicle_categories, vehicles, contracts
WHERE contracts.contr_ID = 3
  AND vehicle_categories.cat_ID = vehicles.veh_cat_ID
  AND contracts.contr_veh_ID = vehicles.veh_ID;
""")

,cat_name
1,Грузовой


Строк: 1


**Запрос 1.2.3.** Соединение JOIN ... ON (INNER JOIN): номера договоров с именами клиентов (первые четыре строки)

In [16]:
q("""
SELECT contracts.contr_number, clients.cl_name
FROM contracts JOIN clients
ON contracts.contr_client_ID = clients.cl_ID
ORDER BY contracts.contr_ID
LIMIT 4;
""")

,contr_number,cl_name
1,Д-2024/001,Лебедев Игорь Олегович
2,Д-2024/002,Сафина Марина Рустамовна
3,Д-2024/003,ООО «Сибтранс»
4,Д-2024/004,Круглов Антон Павлович


Строк: 4


**Запрос 1.2.4.** Соединение с группировкой: число транспортных средств в каждой категории

In [17]:
q("""
SELECT vehicles.veh_cat_ID AS "Код категории",
       COUNT(vehicles.veh_ID) AS "Число ТС"
FROM vehicle_categories JOIN vehicles
ON vehicle_categories.cat_ID = vehicles.veh_cat_ID
GROUP BY vehicles.veh_cat_ID
ORDER BY vehicles.veh_cat_ID;
""")

,Код категории,Число ТС
1,1,4
2,2,3
3,3,2
4,4,1


Строк: 4


<a id="part-3"></a>

## 3. Внешние соединения

**Запрос 1.3.0.** Добавить в таблицу clients двух клиентов без договоров (потом их можно будет удалить)

In [18]:
q("""
INSERT INTO clients (cl_ID, cl_type, cl_name, cl_doc, cl_phone, cl_email, cl_discount)
       VALUES (9,  'Ф', 'Сидоров Виктор Владимирович', '5207 100200', '+79139990019', NULL, 0),
              (10, 'Ф', 'Жуков Павел Денисович',       '5208 300400', '+79139990020', NULL, 0);
""")

INSERT 0 2


**Запрос 1.3.1.** Внутреннее соединение: список клиентов и оформленных ими договоров

In [19]:
q("""
SELECT c.cl_ID, c.cl_name, k.contr_ID
FROM clients c JOIN contracts k
ON c.cl_ID = k.contr_client_ID
ORDER BY c.cl_ID, k.contr_ID;
""")

,cl_id,cl_name,contr_id
1,1,Лебедев Игорь Олегович,1
2,1,Лебедев Игорь Олегович,9
3,2,Сафина Марина Рустамовна,2
4,2,Сафина Марина Рустамовна,10
5,3,ООО «Сибтранс»,3
6,4,Круглов Антон Павлович,4
7,5,Нечаева Елена Борисовна,5
8,6,АО «Омскстрой»,6
9,7,Тимофеев Роман Андреевич,7
10,8,Жданова Полина Сергеевна,8


Строк: 10


**Запрос 1.3.2.** Левое внешнее соединение: список клиентов и их договоров, включая клиентов без договоров

In [20]:
q("""
SELECT c.cl_ID, c.cl_name, k.contr_ID
FROM clients c LEFT JOIN contracts k
ON c.cl_ID = k.contr_client_ID
ORDER BY c.cl_ID, k.contr_ID;
""")

,cl_id,cl_name,contr_id
1,1,Лебедев Игорь Олегович,1.0
2,1,Лебедев Игорь Олегович,9.0
3,2,Сафина Марина Рустамовна,2.0
4,2,Сафина Марина Рустамовна,10.0
5,3,ООО «Сибтранс»,3.0
6,4,Круглов Антон Павлович,4.0
7,5,Нечаева Елена Борисовна,5.0
8,6,АО «Омскстрой»,6.0
9,7,Тимофеев Роман Андреевич,7.0
10,8,Жданова Полина Сергеевна,8.0


Строк: 12


**Запрос 1.3.3.** Правое внешнее соединение: тот же результат, таблица clients — правая

In [21]:
q("""
SELECT c.cl_ID, c.cl_name, k.contr_ID
FROM contracts k RIGHT JOIN clients c
ON c.cl_ID = k.contr_client_ID
ORDER BY c.cl_ID, k.contr_ID;
""")

,cl_id,cl_name,contr_id
1,1,Лебедев Игорь Олегович,1.0
2,1,Лебедев Игорь Олегович,9.0
3,2,Сафина Марина Рустамовна,2.0
4,2,Сафина Марина Рустамовна,10.0
5,3,ООО «Сибтранс»,3.0
6,4,Круглов Антон Павлович,4.0
7,5,Нечаева Елена Борисовна,5.0
8,6,АО «Омскстрой»,6.0
9,7,Тимофеев Роман Андреевич,7.0
10,8,Жданова Полина Сергеевна,8.0


Строк: 12


**Запрос 1.3.4.** Полное внешнее соединение FULL JOIN: клиенты и договоры; результат совпадает с правым соединением, так как каждый договор имеет клиента

In [22]:
q("""
SELECT c.cl_ID, c.cl_name, k.contr_ID
FROM clients c FULL JOIN contracts k
ON c.cl_ID = k.contr_client_ID
ORDER BY c.cl_ID, k.contr_ID;
""")

,cl_id,cl_name,contr_id
1,1,Лебедев Игорь Олегович,1.0
2,1,Лебедев Игорь Олегович,9.0
3,2,Сафина Марина Рустамовна,2.0
4,2,Сафина Марина Рустамовна,10.0
5,3,ООО «Сибтранс»,3.0
6,4,Круглов Антон Павлович,4.0
7,5,Нечаева Елена Борисовна,5.0
8,6,АО «Омскстрой»,6.0
9,7,Тимофеев Роман Андреевич,7.0
10,8,Жданова Полина Сергеевна,8.0


Строк: 12


**Запрос 1.3.5.** Список клиентов, которые не оформляли договоры (несовпадающие строки возвращаются как NULL)

In [23]:
q("""
SELECT c.cl_ID, c.cl_name
FROM contracts k RIGHT OUTER JOIN clients c
ON c.cl_ID = k.contr_client_ID
WHERE k.contr_ID IS NULL;
""")

,cl_id,cl_name
1,10,Жуков Павел Денисович
2,9,Сидоров Виктор Владимирович


Строк: 2


**Запрос 1.3.6.** Удалить добавленных клиентов

In [24]:
q("""
DELETE FROM clients WHERE cl_ID IN (9, 10);
""")

DELETE 2


<a id="part-4"></a>

## 4. Рефлексивные соединения

**Запрос 1.4.1.** Число подчинённых у каждого руководителя (рекурсивная связь emp_chief_ID — emp_ID)

In [25]:
q("""
SELECT parents.emp_name AS руководитель,
       COUNT(children.emp_name) AS "число подчинённых"
FROM employees AS parents INNER JOIN employees AS children
ON parents.emp_ID = children.emp_chief_ID
GROUP BY руководитель
ORDER BY "число подчинённых" DESC;
""")

,руководитель,число подчинённых
1,Гордеев Пётр Ильич,4


Строк: 1


**Запрос 1.4.2.** Определить, какие сотрудники занимают ту же должность, что и сотрудник «Ветров Илья Романович»

In [26]:
q("""
SELECT e2.emp_name
FROM employees e1 JOIN employees e2
ON e1.emp_name = 'Ветров Илья Романович'
AND e1.emp_position = e2.emp_position;
""")

,emp_name
1,Ветров Илья Романович
2,Пащенко Ольга Ивановна


Строк: 2


<a id="part-5"></a>

## 5. Вложенные запросы

**Запрос 1.5.1.** Некоррелированный вложенный запрос: названия и цены тарифов для категории «Грузовой»

In [27]:
q("""
SELECT tar_name, tar_price
FROM tariffs
WHERE tar_cat_ID = (SELECT cat_ID
                    FROM vehicle_categories
                    WHERE cat_name = 'Грузовой')
ORDER BY tar_price;
""")

,tar_name,tar_price
1,"Месячный, грузовой",9000.00


Строк: 1


**Запрос 1.5.2.** Вложенный запрос, возвращающий одно значение: категория, к которой относится самый дорогой тариф

In [28]:
q("""
SELECT cat_name AS Категория
FROM vehicle_categories
WHERE cat_ID = (SELECT tar_cat_ID
                FROM tariffs
                WHERE tar_price = (SELECT MAX(tar_price)
                                   FROM tariffs));
""")

,Категория
1,Грузовой


Строк: 1


**Запрос 1.5.3.** Ключевое слово ANY: имена клиентов, оформивших хотя бы один договор

In [29]:
q("""
SELECT cl_name
FROM clients
WHERE cl_ID = ANY(SELECT contr_client_ID FROM contracts);
""")

,cl_name
1,Лебедев Игорь Олегович
2,Сафина Марина Рустамовна
3,ООО «Сибтранс»
4,Нечаева Елена Борисовна
5,АО «Омскстрой»
6,Жданова Полина Сергеевна
7,Круглов Антон Павлович
8,Тимофеев Роман Андреевич


Строк: 8


**Запрос 1.5.4.** Ключевое слово ALL: платежи, сумма которых превышает среднюю сумму платежей по каждому из договоров

In [30]:
q("""
SELECT pay_receipt_no, pay_sum
FROM payments
WHERE pay_sum > ALL(SELECT AVG(pay_sum) FROM payments
GROUP BY pay_contr_ID)
ORDER BY pay_sum;
""")

,pay_receipt_no,pay_sum
1,КВ-000120,50000.00


Строк: 1


**Запрос 1.5.5.** EXISTS: список клиентов, оформивших хотя бы один договор (переписанный запрос 5.3)

In [31]:
q("""
SELECT cl_name
FROM clients
WHERE EXISTS(SELECT * FROM contracts
             WHERE contracts.contr_client_ID = clients.cl_ID);
""")

,cl_name
1,Лебедев Игорь Олегович
2,Сафина Марина Рустамовна
3,ООО «Сибтранс»
4,Нечаева Елена Борисовна
5,АО «Омскстрой»
6,Жданова Полина Сергеевна
7,Круглов Антон Павлович
8,Тимофеев Роман Андреевич


Строк: 8


**Запрос 1.5.6.** Коррелированный вложенный запрос: клиенты, оформившие больше одного договора (кандидаты на скидку)

In [32]:
q("""
SELECT cl_ID, cl_name FROM clients c
WHERE 1 < (SELECT COUNT(k.contr_client_ID)
           FROM contracts k
           WHERE c.cl_ID = k.contr_client_ID);
""")

,cl_id,cl_name
1,1,Лебедев Игорь Олегович
2,2,Сафина Марина Рустамовна


Строк: 2


<a id="part-6"></a>

## 6. Пример выполнения задания (стр. 117)

**Запрос 1.6.1.** Многотабличный запрос: коды клиентов, оформивших два договора и более

In [33]:
q("""
SELECT clients.cl_ID, COUNT(contracts.contr_ID) AS total
FROM clients LEFT JOIN contracts
ON clients.cl_ID = contracts.contr_client_ID
GROUP BY clients.cl_ID
HAVING COUNT(contracts.contr_ID) >= 2
ORDER BY total DESC, clients.cl_ID ASC;
""")

,cl_id,total
1,1,2
2,2,2


Строк: 2


**Запрос 1.6.2.** Запрос с вложенным запросом: перечень машино-мест, на которые никогда не оформлялись договоры

In [34]:
q("""
SELECT pl_ID, pl_sector, pl_number, pl_status
FROM places
WHERE NOT EXISTS
      (SELECT * FROM contracts
       WHERE contracts.contr_place_ID = places.pl_ID);
""")

,pl_id,pl_sector,pl_number,pl_status
1,12,D,1,ремонт
2,10,C,2,свободно


Строк: 2


## Завершение

Выполните эту ячейку, чтобы завершить транзакцию и закрыть подключение. При ошибке изменения текущей транзакции отменяются; повторите ноутбук с начала.

In [35]:
conn.rollback()
conn.close()
print("Временные изменения отменены.")

Временные изменения отменены.
